# MNIST Reconocimiento de Números
Entrena una red neuronal y usa HTML para dibujar números.

In [1]:
import math
import numpy as np
import logging
import tensorflow as tf
import tensorflow_datasets as tfds

In [2]:
logger=tf.get_logger()
logger.setLevel(logging.ERROR)

In [3]:
dataset,metadata=tfds.load('mnist',as_supervised=True,with_info=True)
train_dataset,test_dataset=dataset['train'],dataset['test']

In [4]:
num_train_examples=metadata.splits['train'].num_examples
num_test_examples=metadata.splits['test'].num_examples

In [5]:
def normalize(images,labels):
 images=tf.cast(images,tf.float32)
 images/=255
 return images,labels

In [6]:
train_dataset=train_dataset.map(normalize)
test_dataset=test_dataset.map(normalize)

In [7]:
model=tf.keras.Sequential([
 tf.keras.layers.Flatten(input_shape=(28,28,1)),
 tf.keras.layers.Dense(64,activation=tf.nn.relu),
 tf.keras.layers.Dense(64,activation=tf.nn.relu),
 tf.keras.layers.Dense(10,activation=tf.nn.softmax)
])

/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [8]:
model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

In [9]:
BATCHSIZE=32
train_dataset=train_dataset.repeat().shuffle(num_train_examples).batch(BATCHSIZE)
test_dataset=test_dataset.batch(BATCHSIZE)

In [10]:
model.fit(train_dataset,epochs=5,steps_per_epoch=math.ceil(num_train_examples/BATCHSIZE))

Epoch 1/5


I0000 00:00:1773189479.265037   98444 tf_record_dataset_op.cc:396] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608


1875/1875 ━━━━━━━━━━━━━━━━━━━━ 2s 612us/step - accuracy: 0.9205 - loss: 0.2729
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 1s 619us/step - accuracy: 0.9634 - loss: 0.1234
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 1s 609us/step - accuracy: 0.9736 - loss: 0.0872
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 1s 608us/step - accuracy: 0.9776 - loss: 0.0705
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 1s 623us/step - accuracy: 0.9812 - loss: 0.0601


In [13]:
test_loss, test_accuracy = model.evaluate(
    test_dataset,
    steps=math.ceil(num_test_examples/32)
)
print(test_accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 529us/step - accuracy: 0.9688 - loss: 0.1075
0.9688000082969666


## Crear archivo HTML para dibujar

In [14]:
html_code = '''
<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<title>MNIST Draw</title>
<script src="https://code.jquery.com/jquery-3.4.1.min.js"></script>
<style>
body { font-family: Arial; text-align:center; }
canvas { border:2px solid black; cursor:crosshair; }
button { margin:10px; padding:10px 20px; }
</style>
</head>
<body>

<h2>Dibuja un número</h2>
<canvas id="canvas" width="280" height="280"></canvas>
<br>
<button onclick="clearCanvas()">Limpiar</button>
<button onclick="sendCanvas()">Predecir</button>

<h3>Resultado: <span id="resultado">-</span></h3>

<script>

var canvas=document.getElementById("canvas");
var ctx=canvas.getContext("2d");

ctx.fillStyle="white";
ctx.fillRect(0,0,canvas.width,canvas.height);

var drawing=false;

canvas.onmousedown=function(){drawing=true;}
canvas.onmouseup=function(){drawing=false;ctx.beginPath();}

canvas.onmousemove=function(e){
 if(!drawing)return;
 ctx.lineWidth=20;
 ctx.lineCap="round";
 ctx.strokeStyle="black";
 ctx.lineTo(e.offsetX,e.offsetY);
 ctx.stroke();
 ctx.beginPath();
 ctx.moveTo(e.offsetX,e.offsetY);
}

function clearCanvas(){
 ctx.fillStyle="white";
 ctx.fillRect(0,0,canvas.width,canvas.height);
}

function sendCanvas(){

 var small=document.createElement("canvas");
 small.width=28;
 small.height=28;

 var smallctx=small.getContext("2d");
 smallctx.drawImage(canvas,0,0,28,28);

 var pixels=[];

 for(var x=0;x<28;x++){
  for(var y=0;y<28;y++){
   var imgData=smallctx.getImageData(y,x,1,1);
   var data=imgData.data;
   var r = data[0];
var g = data[1];
var b = data[2];

var gray = (r + g + b) / 3;

var color = (255 - gray) / 255;
   color=(Math.round(color*100)/100).toFixed(2);
   pixels.push(color);
  }
 }

 $.post("http://localhost:8000",{pixeles:pixels.join(",")},function(response){
  console.log("Resultado:",response);
  document.getElementById("resultado").innerHTML=response;
 });

}

</script>
</body>
</html>
'''
with open('index.html','w',encoding='utf-8') as f:
 f.write(html_code)
print('index.html creado')

index.html creado


## Servidor HTTP para predicción

In [15]:
from urllib import parse
from http.server import HTTPServer,BaseHTTPRequestHandler

In [16]:
class SimpleHTTPRequestHandler(BaseHTTPRequestHandler):
 def do_POST(self):
  content_length=int(self.headers['Content-Length'])
  data=self.rfile.read(content_length)
  data=data.decode().replace('pixeles=','')
  data=parse.unquote(data)

  arr=np.fromstring(data,np.float32,sep=',')
  arr=arr.reshape(28,28)
  arr=arr.reshape(1,28,28,1)

  prediction_values=model.predict(arr,batch_size=1)
  prediction=str(np.argmax(prediction_values))

  self.send_response(200)
  self.send_header('Access-Control-Allow-Origin','*')
  self.end_headers()
  self.wfile.write(prediction.encode())

In [ ]:
print('Servidor iniciado en http://localhost:8000')
server=HTTPServer(('localhost',8000),SimpleHTTPRequestHandler)
server.serve_forever()

Servidor iniciado en http://localhost:8000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step


127.0.0.1 - - [10/Mar/2026 18:41:27] "POST / HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step


127.0.0.1 - - [10/Mar/2026 18:41:34] "POST / HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step


...

127.0.0.1 - - [10/Mar/2026 18:42:02] "POST / HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step


127.0.0.1 - - [10/Mar/2026 18:42:03] "POST / HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step


127.0.0.1 - - [10/Mar/2026 18:42:08] "POST / HTTP/1.1" 200 -
